# DIGITRA — Final ve Temiz Model Eğitimi

Bu defter yalnız üretimde kullanılan son boru hattını içerir.

- 28 öğrenilen sınıf: A-Z, del, space
- El bulunamadığında kural tabanlı sınıf: nothing
- 422 boyutlu kanonik el geometrisi
- Eğitim, doğrulama ve test kullanıcıları tamamen ayrıdır
- Doğrulama kullanıcıları: 10 ve 11
- Kilitli test kullanıcıları: 14 ve 15
- Model: 3 GeometryMLP + GraphGeometryNet + ExtraTrees + 8 ikili uzman

Başlangıç ölçümü: doğrulama %89,56; kilitli kullanıcı testi el algılanan karelerde %81,76; algılama hataları dahil %79,92. %95 iddiası yoktur.


In [8]:
# 1/6 — Ortam ve doğrulanmış veri önbelleği
from pathlib import Path
import copy, hashlib, importlib.util, json, os, random, shutil, sys, time, zipfile, subprocess
try:
    import mediapipe as mp
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "mediapipe==1.0.1"])
    import mediapipe as mp

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from torch.utils.data import DataLoader, TensorDataset

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

for archive_candidate in Path("/content").glob("DIGITRA_TRAINING_KIT_V1*.zip"):
    with zipfile.ZipFile(archive_candidate) as archive_handle:
        archive_handle.extractall("/content")

candidates = list(Path("/content").rglob("DIGITRA_TRAINING_DATA_V1.npz"))
if not candidates:
    raise FileNotFoundError(
        "DIGITRA_TRAINING_DATA_V1.npz bulunamadı. Eğitim kitindeki NPZ dosyasını Colab /content alanına yükleyin."
    )
DATA_PATH = candidates[0]
KIT_ROOT = DATA_PATH.parent
data = np.load(DATA_PATH, allow_pickle=False)

X_train = data["X_train"].astype(np.float32)
y_train = data["y_train"].astype(np.int32)
source_train = data["source_train"].astype(str)
X_val = data["X_val"].astype(np.float32)
y_val = data["y_val"].astype(np.int32)
source_val = data["source_val"].astype(str)
X_test = data["X_test"].astype(np.float32)
y_test = data["y_test"].astype(np.int32)
source_test = data["source_test"].astype(str)
CLASSES = data["classes"].astype(str).tolist()
TEST_INPUT_TOTAL = int(data["test_input_total"][0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("DATA", DATA_PATH)
print("CLASSES", len(CLASSES), CLASSES)


DEVICE cuda NVIDIA A100-SXM4-40GB
DATA /content/DIGITRA_TRAINING_DATA_V1.npz
CLASSES 28 ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'space']


In [3]:
# 2/6 — Veri sözleşmesi ve kullanıcı sızıntısı denetimi
assert X_train.ndim == X_val.ndim == X_test.ndim == 2
assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1] == 422
assert len(CLASSES) == 28
assert np.isfinite(X_train).all() and np.isfinite(X_val).all() and np.isfinite(X_test).all()
assert y_train.min() >= 0 and y_train.max() < len(CLASSES)
assert y_val.min() >= 0 and y_val.max() < 26
assert y_test.min() >= 0 and y_test.max() < 26
assert set(source_train).isdisjoint(set(source_val))
assert set(source_train).isdisjoint(set(source_test))
assert set(source_val).isdisjoint(set(source_test))

def class_counts(y):
    counts = np.bincount(y, minlength=len(CLASSES))
    return {name: int(counts[i]) for i, name in enumerate(CLASSES)}

print("TRAIN", X_train.shape, "sources", sorted(set(source_train)))
print("VAL", X_val.shape, "sources", sorted(set(source_val)))
print("TEST", X_test.shape, "sources", sorted(set(source_test)))
print("TRAIN_COUNTS", class_counts(y_train))
print("VAL_AZ_RANGE", int(np.bincount(y_val, minlength=28)[:26].min()), int(np.bincount(y_val, minlength=28)[:26].max()))
print("TEST_AZ_RANGE", int(np.bincount(y_test, minlength=28)[:26].min()), int(np.bincount(y_test, minlength=28)[:26].max()))
print("LEAKAGE_AUDIT_OK")


TRAIN (31855, 422) sources [np.str_('A'), np.str_('BIG'), np.str_('CONTROL_A'), np.str_('D'), np.str_('RAHUL'), np.str_('SIGNER_0'), np.str_('SIGNER_1'), np.str_('SIGNER_12'), np.str_('SIGNER_13'), np.str_('SIGNER_2'), np.str_('SIGNER_3'), np.str_('SIGNER_4'), np.str_('SIGNER_5'), np.str_('SIGNER_6'), np.str_('SIGNER_7'), np.str_('SIGNER_8'), np.str_('SIGNER_9')]
VAL (5200, 422) sources [np.str_('SIGNER_10'), np.str_('SIGNER_11')]
TEST (5083, 422) sources [np.str_('SIGNER_14'), np.str_('SIGNER_15')]
TRAIN_COUNTS {'A': 1314, 'B': 1324, 'C': 1095, 'D': 1005, 'E': 1012, 'F': 1018, 'G': 999, 'H': 1039, 'I': 976, 'J': 835, 'K': 1317, 'L': 1349, 'M': 1308, 'N': 1266, 'O': 1335, 'P': 1225, 'Q': 1254, 'R': 1262, 'S': 1326, 'T': 1262, 'U': 1197, 'V': 1240, 'W': 1142, 'X': 1003, 'Y': 1002, 'Z': 929, 'del': 935, 'space': 886}
VAL_AZ_RANGE 200 200
TEST_AZ_RANGE 162 200
LEAKAGE_AUDIT_OK


In [4]:
# 3/6 — Son model mimarileri ve eğitim yordamları
HAND_EDGES = [
    (0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),(0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
]

class GeometryMLP(nn.Module):
    def __init__(self, n_in=422, n_out=28):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in,1024), nn.BatchNorm1d(1024), nn.GELU(), nn.Dropout(0.18),
            nn.Linear(1024,512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.16),
            nn.Linear(512,256), nn.BatchNorm1d(256), nn.GELU(), nn.Dropout(0.12),
            nn.Linear(256,n_out),
        )
    def forward(self, x):
        return self.net(x)

adj = np.eye(21, dtype=np.float32)
for a, b in HAND_EDGES:
    adj[a,b] = adj[b,a] = 1.0
degree = np.sum(adj, axis=1)
adj = adj / np.sqrt(degree[:,None] * degree[None,:])

class GraphGeometryNet(nn.Module):
    def __init__(self, n_out=28):
        super().__init__()
        self.register_buffer("adj", torch.tensor(adj))
        self.node_enc = nn.Sequential(nn.Linear(6,128), nn.LayerNorm(128), nn.GELU())
        self.g1 = nn.Sequential(nn.Linear(128,192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(0.10))
        self.g2 = nn.Sequential(nn.Linear(192,192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(0.10))
        self.attn = nn.Linear(192,1)
        self.global_branch = nn.Sequential(
            nn.Linear(296,384), nn.BatchNorm1d(384), nn.GELU(), nn.Dropout(0.15),
            nn.Linear(384,256), nn.GELU(),
        )
        self.head = nn.Sequential(
            nn.Linear(192*3+256,512), nn.BatchNorm1d(512), nn.GELU(), nn.Dropout(0.17),
            nn.Linear(512,256), nn.GELU(), nn.Dropout(0.10), nn.Linear(256,n_out),
        )
    def forward(self, x):
        ci = x[:,:63].reshape(-1,21,3)
        cw = x[:,63:126].reshape(-1,21,3)
        h = self.node_enc(torch.cat([ci,cw], dim=2))
        h = self.g1(torch.einsum("ij,bjd->bid", self.adj, h))
        h = h + self.g2(torch.einsum("ij,bjd->bid", self.adj, h))
        w = torch.softmax(self.attn(h), dim=1)
        pooled = torch.cat([h.mean(1), h.amax(1), (w*h).sum(1)], dim=1)
        glob = self.global_branch(x[:,126:])
        return self.head(torch.cat([pooled,glob], dim=1))

scaler = StandardScaler()
X_train_z = scaler.fit_transform(X_train).astype(np.float32)
X_val_z = scaler.transform(X_val).astype(np.float32)
X_test_z = scaler.transform(X_test).astype(np.float32)

def train_geometry(seed, selected_epoch, schedule_epochs=80):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    model = GeometryMLP().to(DEVICE)
    dataset = TensorDataset(torch.from_numpy(X_train_z), torch.from_numpy(y_train).long())
    loader = DataLoader(
        dataset, batch_size=768, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=2, pin_memory=DEVICE.type == "cuda", drop_last=True,
    )
    counts = np.bincount(y_train, minlength=28).astype(np.float32)
    weights = np.clip(np.sqrt(counts.mean() / np.maximum(counts, 1)), 0.65, 2.5)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, device=DEVICE), label_smoothing=0.035
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=2.5e-3, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=3e-3, epochs=schedule_epochs, steps_per_epoch=len(loader),
        pct_start=0.12, div_factor=10, final_div_factor=100,
    )
    for epoch in range(selected_epoch):
        model.train()
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            xb = xb + torch.randn_like(xb) * 0.018
            xb = xb.masked_fill(torch.rand_like(xb) < 0.012, 0.0)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            scheduler.step()
        print("GEOMETRY", seed, "epoch", epoch + 1, "of", selected_epoch, flush=True)
    model.eval()
    return model

def train_graph(seed=5521, selected_epoch=5, schedule_epochs=90):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    model = GraphGeometryNet().to(DEVICE)
    dataset = TensorDataset(torch.from_numpy(X_train_z), torch.from_numpy(y_train).long())
    loader = DataLoader(
        dataset, batch_size=640, shuffle=True,
        generator=torch.Generator().manual_seed(seed),
        num_workers=2, pin_memory=DEVICE.type == "cuda", drop_last=True,
    )
    counts = np.bincount(y_train, minlength=28).astype(np.float32)
    weights = np.clip(np.sqrt(counts.mean() / np.maximum(counts, 1)), 0.65, 2.5)
    criterion = nn.CrossEntropyLoss(
        weight=torch.tensor(weights, device=DEVICE), label_smoothing=0.025
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.8e-3, weight_decay=2.5e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=2.4e-3, epochs=schedule_epochs, steps_per_epoch=len(loader),
        pct_start=0.15, div_factor=8, final_div_factor=80,
    )
    for epoch in range(selected_epoch):
        model.train()
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            xb = xb + torch.randn_like(xb) * 0.012
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 4.0)
            optimizer.step()
            scheduler.step()
        print("GRAPH", seed, "epoch", epoch + 1, "of", selected_epoch, flush=True)
    model.eval()
    return model


In [5]:
# 4/6 — Modelleri sıfırdan eğit
start = time.time()

extra_trees = ExtraTreesClassifier(
    n_estimators=600,
    criterion="entropy",
    max_features=0.35,
    min_samples_leaf=1,
    class_weight="balanced",
    n_jobs=-1,
    random_state=2026,
)
extra_trees.fit(X_train, y_train)
print("EXTRA_TREES_DONE")

geometry_models = [
    train_geometry(2026, selected_epoch=20),
    train_geometry(3407, selected_epoch=15),
    train_geometry(9913, selected_epoch=26),
]
graph_model = train_graph(5521, selected_epoch=5)
print("BASE_TRAINING_DONE_MIN", round((time.time() - start) / 60, 2))


EXTRA_TREES_DONE
GEOMETRY 2026 epoch 1 of 20
GEOMETRY 2026 epoch 2 of 20
GEOMETRY 2026 epoch 3 of 20
GEOMETRY 2026 epoch 4 of 20
GEOMETRY 2026 epoch 5 of 20
GEOMETRY 2026 epoch 6 of 20
GEOMETRY 2026 epoch 7 of 20
GEOMETRY 2026 epoch 8 of 20
GEOMETRY 2026 epoch 9 of 20
GEOMETRY 2026 epoch 10 of 20
GEOMETRY 2026 epoch 11 of 20
GEOMETRY 2026 epoch 12 of 20
GEOMETRY 2026 epoch 13 of 20
GEOMETRY 2026 epoch 14 of 20
GEOMETRY 2026 epoch 15 of 20
GEOMETRY 2026 epoch 16 of 20
GEOMETRY 2026 epoch 17 of 20
GEOMETRY 2026 epoch 18 of 20
GEOMETRY 2026 epoch 19 of 20
GEOMETRY 2026 epoch 20 of 20
GEOMETRY 3407 epoch 1 of 15
GEOMETRY 3407 epoch 2 of 15
GEOMETRY 3407 epoch 3 of 15
GEOMETRY 3407 epoch 4 of 15
GEOMETRY 3407 epoch 5 of 15
GEOMETRY 3407 epoch 6 of 15
GEOMETRY 3407 epoch 7 of 15
GEOMETRY 3407 epoch 8 of 15
GEOMETRY 3407 epoch 9 of 15
GEOMETRY 3407 epoch 10 of 15
GEOMETRY 3407 epoch 11 of 15
GEOMETRY 3407 epoch 12 of 15
GEOMETRY 3407 epoch 13 of 15
GEOMETRY 3407 epoch 14 of 15
GEOMETRY 3407 e

In [6]:
# 5/6 — Kilitli uzmanlar ve tamamen ayrı kullanıcılarla ölçüm
def base_probabilities(raw, scaled):
    tensor = torch.from_numpy(scaled).to(DEVICE)
    with torch.inference_mode():
        p_geometry = np.mean([
            torch.softmax(model(tensor), dim=1).cpu().numpy()
            for model in geometry_models
        ], axis=0)
        p_graph = torch.softmax(graph_model(tensor), dim=1).cpu().numpy()
    p_et_small = extra_trees.predict_proba(raw)
    p_et = np.zeros((len(raw), len(CLASSES)), dtype=np.float64)
    p_et[:, np.asarray(extra_trees.classes_, dtype=int)] = p_et_small
    return 0.8075 * p_geometry + 0.0425 * p_et + 0.15 * p_graph

expert_configs = [
    ("M","T","raw_subset",np.arange(63,126),60.0,0.0008),
    ("P","Q","raw_subset",np.arange(0,63),1.0,0.0008),
    ("I","J","raw_subset",np.arange(0,63),1.0,0.0008),
    ("B","F","raw_subset",np.arange(0,63),3.0,0.0064),
    ("O","Q","scaled_all",None,10.0,0.001184834123222749),
    ("C","W","scaled_all",None,1.5,0.002369668246445498),
    ("N","T","scaled_all",None,5.0,0.004739336492890996),
    ("K","U","scaled_all",None,5.0,0.002369668246445498),
]

LOCKED_PAIRS = []
expert_specs = {}
for left_name, right_name, mode, indices, c_value, gamma_value in expert_configs:
    pair = (CLASSES.index(left_name), CLASSES.index(right_name))
    train_mask = np.isin(y_train, pair)
    if mode == "raw_subset":
        model = make_pipeline(
            StandardScaler(),
            SVC(
                C=c_value, gamma=gamma_value, kernel="rbf",
                class_weight="balanced", cache_size=3000,
            ),
        )
        model.fit(X_train[train_mask][:, indices], y_train[train_mask])
    else:
        model = SVC(
            C=c_value, gamma=gamma_value, kernel="rbf",
            class_weight="balanced", cache_size=3000,
        )
        model.fit(X_train_z[train_mask], y_train[train_mask])
    LOCKED_PAIRS.append(pair)
    expert_specs[pair] = {"model": model, "mode": mode, "idx": indices}
    print("EXPERT_DONE", left_name, right_name)

def apply_experts(prediction, raw, scaled):
    output = prediction.copy()
    for pair in LOCKED_PAIRS:
        active = np.isin(output, pair)
        if not active.any():
            continue
        spec = expert_specs[pair]
        expert_input = (
            raw[active][:, spec["idx"]]
            if spec["mode"] == "raw_subset"
            else scaled[active]
        )
        output[active] = spec["model"].predict(expert_input)
    return output

val_probability = base_probabilities(X_val, X_val_z)
val_prediction = apply_experts(val_probability.argmax(1), X_val, X_val_z)
VAL_ACCURACY = float(accuracy_score(y_val, val_prediction))
VAL_MACRO_F1 = float(f1_score(y_val, val_prediction, labels=list(range(26)), average="macro", zero_division=0))

test_probability = base_probabilities(X_test, X_test_z)
test_prediction = apply_experts(test_probability.argmax(1), X_test, X_test_z)
TEST_DETECTED_ACCURACY = float(accuracy_score(y_test, test_prediction))
TEST_MACRO_F1 = float(f1_score(y_test, test_prediction, labels=list(range(26)), average="macro", zero_division=0))
TEST_END_TO_END = float((test_prediction == y_test).sum() / TEST_INPUT_TOTAL)

print("VALIDATION_UNSEEN_SIGNERS", {"accuracy": VAL_ACCURACY, "macro_f1": VAL_MACRO_F1})
print("LOCKED_TEST_UNSEEN_SIGNERS", {
    "detected_accuracy": TEST_DETECTED_ACCURACY,
    "macro_f1": TEST_MACRO_F1,
    "end_to_end_accuracy": TEST_END_TO_END,
    "detector_misses": int(TEST_INPUT_TOTAL - len(y_test)),
})
for signer in sorted(set(source_test)):
    mask = source_test == signer
    print("TEST_SIGNER", signer, "accuracy", float(accuracy_score(y_test[mask], test_prediction[mask])))


EXPERT_DONE M T
EXPERT_DONE P Q
EXPERT_DONE I J
EXPERT_DONE B F
EXPERT_DONE O Q
EXPERT_DONE C W
EXPERT_DONE N T
EXPERT_DONE K U
VALIDATION_UNSEEN_SIGNERS {'accuracy': 0.895576923076923, 'macro_f1': 0.903470931812225}
LOCKED_TEST_UNSEEN_SIGNERS {'detected_accuracy': 0.8176273854023215, 'macro_f1': 0.8104509230993219, 'end_to_end_accuracy': 0.7992307692307692, 'detector_misses': 117}
TEST_SIGNER SIGNER_14 accuracy 0.8438461538461538
TEST_SIGNER SIGNER_15 accuracy 0.7901731776077325


In [9]:
# 6/6 — Sürüm paketi, bütünlük özeti ve yükleme testi
RELEASE_DIR = Path("/content/DIGITRA_LANDMARK_V1")
if RELEASE_DIR.exists():
    shutil.rmtree(RELEASE_DIR)
RELEASE_DIR.mkdir(parents=True)

torch.save({
    "classes": CLASSES,
    "geometry_states": [model.state_dict() for model in geometry_models],
    "graph_state": graph_model.state_dict(),
    "input_dim": 422,
    "feature_version": "canonical_422_v1",
}, RELEASE_DIR / "neural_ensemble.pt")
joblib.dump(scaler, RELEASE_DIR / "feature_scaler.joblib", compress=3)
joblib.dump(extra_trees, RELEASE_DIR / "extra_trees.joblib", compress=3)
joblib.dump({"pairs": LOCKED_PAIRS, "specs": expert_specs}, RELEASE_DIR / "pair_experts.joblib", compress=3)
shutil.copy2(KIT_ROOT / "inference.py", RELEASE_DIR / "inference.py")
shutil.copy2(KIT_ROOT / "hand_landmarker.task", RELEASE_DIR / "hand_landmarker.task")

metadata = {
    "name": "Digitra Landmark Ensemble V1",
    "classes": CLASSES,
    "no_hand_class": "nothing",
    "validation": {
        "signers": ["10", "11"],
        "detected_accuracy": VAL_ACCURACY,
        "macro_f1": VAL_MACRO_F1,
    },
    "locked_test": {
        "signers": ["14", "15"],
        "detected_accuracy": TEST_DETECTED_ACCURACY,
        "macro_f1": TEST_MACRO_F1,
        "end_to_end_accuracy": TEST_END_TO_END,
        "detector_misses": int(TEST_INPUT_TOTAL - len(y_test)),
    },
    "training": {
        "feature_rows": int(len(X_train)),
        "feature_version": "canonical_422_v1",
        "signer_disjoint": True,
    },
    "blend": {"geometry": 0.8075, "extra_trees": 0.0425, "graph": 0.15},
    "experts": [[CLASSES[a], CLASSES[b]] for a, b in LOCKED_PAIRS],
    "seed": SEED,
}
(RELEASE_DIR / "metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
(RELEASE_DIR / "requirements.txt").write_text(
    "\n".join([
        f"numpy=={np.__version__}",
        f"torch=={torch.__version__.split('+')[0]}",
        f"scikit-learn=={sklearn.__version__}",
        f"joblib=={joblib.__version__}",
        "Pillow",
        "mediapipe",
    ]) + "\n",
    encoding="utf-8",
)
pd.DataFrame({
    "true": [CLASSES[i] for i in y_test],
    "pred": [CLASSES[i] for i in test_prediction],
    "signer": source_test,
}).to_csv(RELEASE_DIR / "locked_test_predictions.csv", index=False)

spec = importlib.util.spec_from_file_location("digitra_export_check", RELEASE_DIR / "inference.py")
module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = module
spec.loader.exec_module(module)
check_model = module.DigitraRecognizer(RELEASE_DIR, device="cpu", initialize_detector=False)
probe_indices = np.linspace(0, len(X_test) - 1, 64, dtype=int)
for index in probe_indices:
    got = check_model.predict_feature(X_test[index])["label"]
    expected = CLASSES[int(test_prediction[index])]
    assert got == expected, (int(index), expected, got)
check_model.close()
print("FEATURE_PARITY_OK", len(probe_indices), "of", len(probe_indices))

manifest = {}
for path in sorted(RELEASE_DIR.iterdir()):
    if path.is_file() and path.name != "MANIFEST.json":
        manifest[path.name] = {
            "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "bytes": path.stat().st_size,
        }
(RELEASE_DIR / "MANIFEST.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)
archive = Path(shutil.make_archive("/content/DIGITRA_LANDMARK_V1_RELEASE", "zip", RELEASE_DIR))
archive_sha = hashlib.sha256(archive.read_bytes()).hexdigest()
Path(str(archive) + ".sha256").write_text(
    f"{archive_sha}  {archive.name}\n", encoding="utf-8"
)
print("RELEASE_READY", archive, f"{archive.stat().st_size/1024/1024:.2f} MB")
print("SHA256", archive_sha)


FEATURE_PARITY_OK 64 of 64
RELEASE_READY /content/DIGITRA_LANDMARK_V1_RELEASE.zip 122.46 MB
SHA256 b1eca8df9f9ee4c083228735ba98108dc26cc83bd247dd1923054408a5cade9c


## Tamamlanma koşulu

Son hücrede FEATURE_PARITY_OK ve RELEASE_READY yazıları görünmeden model teslim edilmiş sayılmaz. Doğrulama ve kilitli test sonuçları ayrı tutulur; test kullanıcıları model veya hiperparametre seçmek için kullanılmaz.

